# Gold Layer — Banking Fraud Analytics

Aggregate validated Silver transactions into business-ready metrics.

**Metrics**
- Total transactions
- Total transaction amount
- Fraudulent transaction count
- Fraud rate percentage

**Grouping:** Merchant and transaction type  
**Input:** Silver Delta  
**Output:** Gold Delta

### Define silver_path in Cell 2

In [0]:
silver_path = (
    "abfss://silver@stbankingfrauddetection.dfs.core.windows.net/"
    "banking_transactions/"
)

### Define Gold paths and aggregation

In [0]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum,
    when, round as spark_round
)

gold_path = (
    "abfss://gold@stbankingfrauddetection.dfs.core.windows.net/"
    "fraud_analytics/"
)

gold_checkpoint = (
    "abfss://gold@stbankingfrauddetection.dfs.core.windows.net/"
    "checkpoints/gold_analytics_v1/"
)

silver_stream = (
    spark.readStream
    .format("delta")
    .load(silver_path)
)

gold_aggregated = (
    silver_stream
    .groupBy("merchant", "transaction_type")
    .agg(
        count("*").alias("total_transactions"),
        spark_sum("amount").alias("total_amount"),
        spark_sum(
            when(col("is_fraud") == True, 1).otherwise(0)
        ).alias("fraud_count")
    )
    .withColumn(
        "fraud_rate_pct",
        spark_round(
            col("fraud_count") / col("total_transactions") * 100,
            2
        )
    )
)

print("Gold aggregation configured.")

Gold aggregation configured.


## Start Gold streaming

In [0]:
gold_query = (
    gold_aggregated.writeStream
    .outputMode("complete")
    .format("delta")
    .option("path", gold_path)
    .option("checkpointLocation", gold_checkpoint)
    .start()
)

print("Gold analytics stream started.")

Gold analytics stream started.


### 4. Expected output

In [0]:
display(
    spark.read
    .format("delta")
    .load(gold_path)
)

merchant,transaction_type,total_transactions,total_amount,fraud_count,fraud_rate_pct
Electronics Store,UNUSUAL_LOCATION,3,15324.279999999999,3,100.0
Online Shopping,UNUSUAL_LOCATION,4,17449.550000000003,4,100.0
Grocery Store,UNUSUAL_LOCATION,7,34142.27,7,100.0
Fuel Station,UNUSUAL_LOCATION,4,25503.480000000003,4,100.0
Electronics Store,HIGH_VALUE,7,521319.76999999996,7,100.0
Restaurant,UNUSUAL_LOCATION,8,37035.05,8,100.0
Online Shopping,HIGH_VALUE,5,454174.66000000003,5,100.0
Grocery Store,HIGH_VALUE,6,599889.21,6,100.0
Electronics Store,NORMAL,97,458020.62000000005,0,0.0
Fuel Station,HIGH_VALUE,5,530699.05,5,100.0


## Pipeline Validation — Gold Streaming Status

Verify that the Gold streaming query is active and has no recorded exception.

In [0]:
print("Gold active:", gold_query.isActive)
print("Gold status:", gold_query.status)
print("Gold exception:", gold_query.exception())

Gold active: True
Gold status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Gold exception: None


# Gold Layer — Final Data Validation

Validate the aggregated Gold Delta dataset before connecting it to Power BI.

In [0]:
gold_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

print("Gold row count:", gold_df.count())
gold_df.printSchema()

display(gold_df.orderBy("merchant", "transaction_type"))

Gold row count: 15
root
 |-- merchant: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- total_transactions: long (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- fraud_count: long (nullable = true)
 |-- fraud_rate_pct: double (nullable = true)



merchant,transaction_type,total_transactions,total_amount,fraud_count,fraud_rate_pct
Electronics Store,HIGH_VALUE,11,1020487.26,11,100.0
Electronics Store,NORMAL,186,917348.05,0,0.0
Electronics Store,UNUSUAL_LOCATION,9,54136.630000000005,9,100.0
Fuel Station,HIGH_VALUE,13,1310455.46,13,100.0
Fuel Station,NORMAL,198,1011093.8,0,0.0
Fuel Station,UNUSUAL_LOCATION,9,46349.83,9,100.0
Grocery Store,HIGH_VALUE,11,1197150.87,11,100.0
Grocery Store,NORMAL,216,1074736.18,0,0.0
Grocery Store,UNUSUAL_LOCATION,15,65751.77,15,100.0
Online Shopping,HIGH_VALUE,7,656831.16,7,100.0


# Gold Layer — Power BI Export

Export the aggregated Gold fraud analytics to a single CSV file for dashboard development.

**Source:** Gold Delta table  
**Destination:** Gold container in ADLS Gen2  
**Format:** CSV with header

In [0]:
powerbi_export_path = (
    "abfss://gold@stbankingfrauddetection.dfs.core.windows.net/"
    "powerbi_export/"
)

(
    gold_df.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(powerbi_export_path)
)

print("Gold CSV export completed.")

Gold CSV export completed.
